In [82]:
import pandas as pd
from data_profiling import ProfileReport
import numpy as np

# Partie 1 Profiling & audit

In [83]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")
df.describe()

/tmp/ipykernel_6181/1620990980.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.362470e+05,227232.000000,227232.000000,227232.000000,227232.000000,134362.000000
mean,6.924200e+08,14.983180,102.212347,2.678738,46.733937,52603.766683
std,2.572975e+08,47.103571,579.594838,4.529209,4.034939,26780.472001
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,5.243353e+08,2.000000,22.000000,0.774255,44.871519,31112.500000
50%,8.427185e+08,4.000000,22.000000,2.412432,47.340547,57160.000000
75%,8.951636e+08,9.000000,100.000000,4.836760,48.865021,76410.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000


In [84]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [85]:
profile = ProfileReport(df, title="Profiling Report")

In [86]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]/home/constantin/Documents/cours/m1/qualite_donnee/coda_qualite_donnees/venv/lib/python3.13/site-packages/data_profiling/visualisation/utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) DejaVu Sans.
  plt.savefig(
/home/constantin/Documents/cours/m1/qualite_donnee/coda_qualite_donnees/venv/lib/python3.13/site-packages/data_profiling/visualisation/utils.py:73: UserWarning: Glyph 136 (\x88) missing from font(s) DejaVu Sans.
  plt.savefig(
/home/constantin/Documents/cours/m1/qualite_donnee/coda_qualite_donnees/venv/lib/python3.13/site-packages/data_profiling/visualisation/utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) DejaVu Sans.
  plt.savefig(
/home/constantin/Documents/cours/m1/qualite_donnee/coda_qualite_donnees/venv/lib/python3.13/site-packages/data_profiling/visualisation/utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) DejaVu Sans.
  plt.savefig(
/home/constantin/Documents/c

Rapport généré : rapport_data_profiling.html


In [87]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.

In [88]:
df["siren_amenageur"] = (
    df["siren_amenageur"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

Le type de "siren_amenageur" était un float, on l'a converti en str pour pouvoir effectuer du Regex dessus et checker sa validité 

In [89]:
audit_rows = []

# -------------------------
# 1. COMPLETUDE
# -------------------------
for col in df.columns:
    missing_rate = df[col].isna().mean()

    audit_rows.append({
        "dimension": "completude",
        "colonne": col,
        "pct_problemes": round(missing_rate * 100, 2),
        "gravite": "haute" if missing_rate > 0.3 else "moyenne" if missing_rate > 0.1 else "faible"
    })


# -------------------------
# 2. UNICITE (ID station)
# -------------------------
dup_rate = df.duplicated(subset=["id_station_itinerance"]).mean()

audit_rows.append({
    "dimension": "unicite",
    "colonne": "id_station_itinerance",
    "pct_problemes": round(dup_rate * 100, 2),
    "gravite": "haute" if dup_rate > 0.05 else "moyenne" if dup_rate > 0.01 else "faible"
})


# -------------------------
# 3. VALIDITE - SIREN
# -------------------------
siren_invalid = ~df["siren_amenageur"].astype(str).str.match(r"^\d{9}$")
siren_rate = siren_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "siren_amenageur",
    "pct_problemes": round(siren_rate * 100, 2),
    "gravite": "haute" if siren_rate > 0.1 else "moyenne" if siren_rate > 0.02 else "faible"
})


# -------------------------
# 4. COHERENCE GEO
# -------------------------
geo_invalid = ~(
    df["consolidated_latitude"].between(41, 51) &
    df["consolidated_longitude"].between(-5, 10)
)

geo_rate = geo_invalid.mean()

audit_rows.append({
    "dimension": "coherence_geo",
    "colonne": "latitude_longitude",
    "pct_problemes": round(geo_rate * 100, 2),
    "gravite": "haute" if geo_rate > 0.2 else "moyenne" if geo_rate > 0.05 else "faible"
})


# -------------------------
# 5. VALIDITE PUISSANCE
# -------------------------
power_invalid = df["puissance_nominale"] <= 0
power_rate = power_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "puissance_nominale",
    "pct_problemes": round(power_rate * 100, 2),
    "gravite": "haute" if power_rate > 0.1 else "moyenne" if power_rate > 0.02 else "faible"
})


# -------------------------
# RESULTAT FINAL
# -------------------------
audit_df = pd.DataFrame(audit_rows)
audit_df.sort_values("pct_problemes", ascending=False)

,dimension,colonne,pct_problemes,gravite
37,completude,observations,79.53,haute
27,completude,tarification,75.20,haute
52,unicite,id_station_itinerance,71.86,haute
35,completude,num_pdl,48.96,haute
39,completude,cable_t2_attache,46.97,haute
47,completude,consolidated_code_postal,40.87,haute
36,completude,date_mise_en_service,40.61,haute
53,validite,siren_amenageur,40.05,haute
34,completude,raccordement,34.90,haute
16,completude,id_pdc_local,34.62,haute


In [90]:
audit_df.groupby("dimension")["pct_problemes"].mean()

dimension
coherence_geo     0.530000
completude       11.496538
unicite          71.860000
validite         21.185000
Name: pct_problemes, dtype: float64

On voit que la catégorie qui pose le plus de problèmes est l'unicité avec 72% de données non uniques par colonnes en moyenne puis quelques problèmes de complétude, validité et de cohérence_geo

# Partie 2 Grain & doublons

In [91]:
df["num_pdl"].isna().sum()

np.int64(111260)

In [92]:
df.sample(10)

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
192200,SOWATT SOLUTIONS,903356970,contact@sowattsolutions.com,SOWATT SOLUTIONS,contact@sowattsolutions.com,04 79 68 52 99,SOWATT SOLUTIONS,FRSWSE1234597489,1234597489,ABB 24kW Majic,...,3550d1a8-210d-4f5b-88ed-7a3128bda91d,sowatt-solutions,2026-06-24T08:26:06.996000+00:00,5.959297,45.840160,74150.0,Rumilly,True,True,False
176451,Le Plein Tarnais,nan,NaN,Bouygues Energies & Services,support@alizecharge.fr,0805021480,Le Plein Tarnais,FRS81E81004016,NaN,ALBI - Commandant Blanche,...,5f29737c-5393-46f9-8140-2509992adc7a,alize,2025-03-17T08:58:10.352000+00:00,2.154160,43.922080,81000.0,Albi,True,True,False
14039,Atlante,911482628,digital@atlante.energy,Atlante France,support.france@atlante.energy,tel:+33-1-83-75-07-25,Atlante - A9 - Aire de service de Marguerittes...,FRATLPFR00310,NaN,Atlante - A9 - Aire de service de Marguerittes...,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,4.439411,43.869269,30320.0,Marguerittes,True,True,False
15845,MISSING,nan,NaN,Atlante | FR*ATL,operations.france@atlante.energy,NaN,Atlante,FRATLP4282316367457871389,1453468,Atlante/FRATLFR00300,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,2.879920,48.941010,NaN,NaN,False,False,False
70100,Grand Lyon,419070180,lucas.malacarne@izivia.com,IZIVIA,sav@izivia.com,+33(0)972668001,Grand Lyon,FRGLYPLYON30,FRSODPLYON301,LY307 - Rue d'Arménie,...,e297f18c-bb35-445f-af43-0217c27ab4fe,izivia,2023-11-23T13:01:13.994000+00:00,4.850437,45.754512,NaN,NaN,False,False,False
136716,Power Dot France,891118473,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,tel:+33-1-76-31-06-84,Power Dot France,FRPD1PMBRSJG,NaN,Mr. Bricolage - Saint-Jouan-des-Guérets,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,-1.971991,48.602492,35430.0,Saint-Jouan-des-Guérets,True,True,False
165564,SYNDICAT DEPARTEMENTAL D'ENERGIE DES COTES D'A...,nan,NaN,Bouygues Energies & Services,support@alizecharge.fr,0805021480,Ouest Charge,FRS22E22319001,NaN,SAINT MICHEL EN GRÈVE - Route d'Arvor,...,5f29737c-5393-46f9-8140-2509992adc7a,alize,2025-03-17T08:58:10.352000+00:00,-3.567519,48.683102,22300.0,Saint-Michel-en-Grève,True,True,False
67950,DSC - Saint-Brieuc,nan,NaN,Freshmile | FR*FR1,roaming@freshmile.com,NaN,Freshmile France,FRFR1P6964200189534355448,1439545,Freshmile France/L4NOHBJZQP,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,-2.742564,48.502402,NaN,NaN,False,False,False
125831,Power Dot France,891118473,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,0176310684,Power Dot France,FRPD1PACCVDA,NaN,hotelF1 - Villeneuve-d'Ascq,...,1bf98bac-94a9-4909-8726-47a203038a40,power-dot-france,2023-01-30T13:44:34.220000+00:00,3.143224,50.642391,59650.0,Villeneuve-d'Ascq,True,True,True
46530,Electra,891624884,help@go-electra.com,Electra,help@go-electra.com,tel:+33-1-86-65-99-99,Electra Senlis - Intermarché Hyper Senlis,FRELCP13010953,FRELCP13010953,Electra Senlis - Intermarché Hyper Senlis,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,2.590281,49.222223,60300.0,Senlis,True,True,False


In [93]:
df['siren_amenageur']

0               nan
1               nan
2               nan
3               nan
4               nan
            ...    
227227    895193019
227228    508013281
227229    444811970
227230    444811970
227231    444811970
Name: siren_amenageur, Length: 227232, dtype: object

In [94]:
columnes_a_verifier = [
    col for col in df.columns
    if "id_pdc" in col.lower()
    or "itiner" in col.lower()
    or "station" in col.lower()
    or "date" in col.lower()
    or "operateur" in col.lower()
]

print("Colonnes candidates:")
print(columnes_a_verifier)

col_id = "id_pdc_itinerance"
print("\nNombre de lignes:", len(df))
print("Valeurs distinctes de id_pdc_itinerance:", df[col_id].nunique(dropna=False))
print("Lignes en doublon sur id_pdc_itinerance:", df.duplicated(subset=[col_id]).sum())

freq_id = df[col_id].value_counts(dropna=False)
print("\nTop 10 des valeurs les plus fréquentes:")
print(freq_id.head(10))

if len(columnes_a_verifier) > 1:
    print("\nTest de granularité sur plusieurs colonnes:")
    for taille in range(2, min(5, len(columnes_a_verifier)) + 1):
        subset = columnes_a_verifier[:taille]
        doublons = df.duplicated(subset=subset).sum()
        print(f"{subset} -> doublons: {doublons}")

Colonnes candidates:
['nom_operateur', 'contact_operateur', 'telephone_operateur', 'id_station_itinerance', 'id_station_local', 'nom_station', 'implantation_station', 'adresse_station', 'id_pdc_itinerance', 'id_pdc_local', 'station_deux_roues', 'date_mise_en_service', 'date_maj', 'consolidated_longitude', 'consolidated_latitude', 'consolidated_code_postal', 'consolidated_commune', 'consolidated_is_lon_lat_correct', 'consolidated_is_code_insee_verified', 'consolidated_is_code_insee_modified']

Nombre de lignes: 227232
Valeurs distinctes de id_pdc_itinerance: 160138
Lignes en doublon sur id_pdc_itinerance: 67094

Top 10 des valeurs les plus fréquentes:
id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRALLEGO8005071      3
FRALLEGO8005072      3
FRUBIE10021422       3
FRALLEGO8005251      3
FRALLEGO8005252      3
FRUBIE10021897       3
FRALLEGO8005271      3
FRALLEGO8005272      3
Name: count, dtype: int64

Test de granularité sur plusieurs colonnes:
['nom_operateur', 'con

In [95]:
id_exemple = freq_id.index[1]
exemple = df.loc[df[col_id] == id_exemple, [
    col for col in [
        "id_station_itinerance",
        "id_pdc_itinerance",
        "id_pdc_local",
        "nom_station",
        "nom_operateur",
        "date_mise_en_service",
        "date_maj",
        "consolidated_commune",
        "consolidated_code_postal",
    ] if col in df.columns
]]
print(f"Exemple pour {id_exemple}:")
display(exemple)

Exemple pour FRLIBE3126085:


,id_station_itinerance,id_pdc_itinerance,id_pdc_local,nom_station,nom_operateur,date_mise_en_service,date_maj,consolidated_commune,consolidated_code_postal
105425,FRLIBE3126085,FRLIBE3126085,frlibe3126085,Résidence Carouge,Bornevo,2023-06-23,2023-10-21,Brétigny-sur-Orge,91220.0
105426,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,NaN,2023-11-10,2023-11-10,NaN,NaN
105427,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0
105428,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0


id_pdc_itinerance n'est pas une clef car une ligne représente un snapshot de l'évolution de la mission d'un opérateur. Donc une itinérance peut apparaitre pour plusieurs dates différentes, nom d'opérateur, point de charge local.

Ce qui nous interesse nous, c'est d'avoir une ligne par point de livraison x point de charge

In [96]:
dup_rate = df.duplicated(subset=["id_pdc_itinerance", "num_pdl"]).mean()

print("Taux de doublons PDC×PDL :", round(dup_rate * 100, 2), "%")

Taux de doublons PDC×PDL : 12.43 %


On a tout de même 12.43% de doublons c'est à dire de lignes qui ont le même couple (PDL,PDC)
Pour être sûr de garder un dataset propre, on ne va garder dans les doublons que ceux de la dernière date de mise à jour

In [97]:
df["date_maj"] = pd.to_datetime(df["date_maj"], errors="coerce")

In [98]:
df_clean = (
    df.sort_values("date_maj", ascending=False)
        .drop_duplicates(
            subset=["id_pdc_itinerance", "num_pdl"],
            keep="first"
        )
)

In [99]:
nb_doublons = df_clean.duplicated(
    subset=["id_pdc_itinerance", "num_pdl"]
).sum()

print(f"Nombre de doublons : {nb_doublons}")

Nombre de doublons : 0


On a 160 138 bornes

# PARTIE 3

In [100]:
def fix_encoding(x):
    if isinstance(x, str):
        return (
            x.encode("latin1", errors="ignore")
             .decode("utf-8", errors="ignore")
        )
    return x

df_clean = df_clean.applymap(fix_encoding)

/tmp/ipykernel_6181/3559430724.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_clean = df_clean.applymap(fix_encoding)


In [101]:
df_clean["consolidated_commune"] = (
    df_clean["consolidated_commune"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.title()
)

In [102]:
df_clean["date_maj"] = pd.to_datetime(df_clean["date_maj"], errors="coerce")
df_clean["date_mise_en_service"] = pd.to_datetime(df_clean["date_mise_en_service"], errors="coerce")
df_clean["created_at"] = pd.to_datetime(df_clean["created_at"], errors="coerce")
df_clean["last_modified"] = pd.to_datetime(df_clean["last_modified"], errors="coerce")

In [103]:
df_clean["puissance_nominale"] = pd.to_numeric(
    df_clean["puissance_nominale"],
    errors="coerce"
)

In [104]:
mask_watts = df_clean["puissance_nominale"] > 1000

df_clean.loc[mask_watts, "puissance_nominale"] /= 1000

In [105]:
df_clean["puissance_nominale"].describe()

count    198977.000000
mean         72.797102
std         101.782091
min           0.000000
25%          22.000000
50%          22.000000
75%         100.000000
max         600.000000
Name: puissance_nominale, dtype: float64

# PARTIE 4

In [30]:
import great_expectations as gx

1. "id_pdc_itinerance" doit être non nulle
>Sans point de charge la ligne n'est plus exploitable

2. "id_station_itinerance" non nulle 
>Un PDC a forcément une station

3. Le subset ["id_pdc_itinerance","num_pdl"] doit être unique
> On prend toujours la dernière date de MàJ

4. "puissance_nominale" est > 0 et <600kW
> Une borne standard fournie entre 0 et 400kW, exceptionnellement 500kW donc 600 permet d'éliminer les valeurs aberrantes 

5. "consolidated_latitude" entre 41 et 52 

6. consolidated_longitude entre -5.5 et 10
>Le dataset concerne les IRVE françaises donc on adapte les coordonnées en fonction

7. Le code postal doit être une suite de 5 chiffres 
>Standard des codes FR

8. Le siren doit être une suite de 9chiffres
>Standard Français

In [34]:
print(gx.__version__)

1.18.2


In [46]:
import great_expectations as gx

context = gx.get_context()

datasource = context.data_sources.add_pandas(
    name="pandas_source"
)

asset = datasource.add_dataframe_asset(
    name="irve_asset"
)

batch_definition = asset.add_batch_definition_whole_dataframe(
    "whole_dataframe"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_clean}
)

In [107]:
expectation = gx.expectations.ExpectCompoundColumnsToBeUnique(
    column_list=["id_pdc_itinerance", "num_pdl"]
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 9/9 [00:00<00:00,  9.39it/s]  

True


In [48]:
expectation = gx.expectations.ExpectCompoundColumnsToBeUnique(
    column_list=["id_pdc_itinerance", "num_pdl"]
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 9/9 [00:01<00:00,  7.44it/s]  

True


In [51]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="consolidated_latitude",
    min_value=41,
    max_value=52
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 15.24it/s] 

False


In [52]:
from scipy.spatial import cKDTree


def to_unit_sphere(latitudes, longitudes):
    latitude_rad = np.deg2rad(latitudes)
    longitude_rad = np.deg2rad(longitudes)
    return np.column_stack(
        (
            np.cos(latitude_rad) * np.cos(longitude_rad),
            np.cos(latitude_rad) * np.sin(longitude_rad),
            np.sin(latitude_rad),
        )
    )


communes_geo = df_communes_france[["code_insee", "latitude_centre", "longitude_centre"]].copy()
communes_geo["code_insee"] = communes_geo["code_insee"].astype(str).str.zfill(5)
communes_geo["latitude_centre"] = pd.to_numeric(communes_geo["latitude_centre"], errors="coerce")
communes_geo["longitude_centre"] = pd.to_numeric(communes_geo["longitude_centre"], errors="coerce")
communes_geo = communes_geo.dropna(subset=["code_insee", "latitude_centre", "longitude_centre"])

station_mask = (
    df_clean["code_insee_commune"].isna()
    & df_clean["consolidated_latitude"].notna()
    & df_clean["consolidated_longitude"].notna()
)

station_coords = df_clean.loc[station_mask, ["consolidated_latitude", "consolidated_longitude"]].copy()
station_coords["consolidated_latitude"] = pd.to_numeric(station_coords["consolidated_latitude"], errors="coerce")
station_coords["consolidated_longitude"] = pd.to_numeric(station_coords["consolidated_longitude"], errors="coerce")
station_coords = station_coords.dropna(subset=["consolidated_latitude", "consolidated_longitude"])

station_xyz = to_unit_sphere(station_coords["consolidated_latitude"], station_coords["consolidated_longitude"])
commune_xyz = to_unit_sphere(communes_geo["latitude_centre"], communes_geo["longitude_centre"])

nearest_communes = cKDTree(commune_xyz).query(station_xyz, k=1)
chord_distance = nearest_communes[0]
nearest_index = nearest_communes[1]

approx_distance_km = 6371.0 * 2 * np.arcsin(np.clip(chord_distance / 2, 0, 1))
matched_codes = communes_geo.iloc[nearest_index]["code_insee"].to_numpy()
matched_index = station_coords.index

df_clean.loc[matched_index, "code_insee_commune"] = matched_codes

print("Valeurs de code_insee_commune complétées :", len(matched_index))
print("Valeurs restantes manquantes :", df_clean["code_insee_commune"].isna().sum())
print("Distance médiane au centroïde communal :", round(float(np.median(approx_distance_km)), 2), "km")
print("Correspondances à plus de 20 km :", int((approx_distance_km > 20).sum()))

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 16.14it/s] 

False


In [56]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="puissance_nominale",
    min_value=0,
    max_value=600
)

result = batch.validate(expectation)

print(result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 13.42it/s] 

False
